# FIREDpy Fire Spread Vector Validation

This notebook validates fire spread speed and vector outputs from a FIREDpy run by loading the latest event-perimeter geopackage and generating visual diagnostics.

> **Target dataset**
> `/Users/dalo2903/firedpy_runs/test_shape/outputs/shapefiles/fired_test_shape_2020_to_2020_daily.gpkg`


## 1) Imports and notebook setup

This section imports analysis libraries and configures plotting style for quick visual QA.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 200)


## 2) Locate and load FIREDpy output

The default path points to the latest daily output geopackage used for vector-speed validation.

If the file is not present in the current environment, update `GPKG_PATH` to your local output path.

In [ ]:
GPKG_PATH = Path('/Users/dalo2903/firedpy_runs/test_shape/outputs/shapefiles/fired_test_shape_2020_to_2020_daily.gpkg')

if not GPKG_PATH.exists():
    warnings.warn(
        f'GeoPackage not found at {GPKG_PATH}. '
        'Update GPKG_PATH to a valid FIREDpy output before running the plotting cells.'
    )

gdf = gpd.read_file(GPKG_PATH) if GPKG_PATH.exists() else gpd.GeoDataFrame()
gdf.head()


## 3) Reuse template plotting utilities (if available)

This notebook attempts to import plotting functions from the operational template notebook (`FIREDpy speed-Operational.ipynb`) to avoid rewriting utility code.

If the template notebook is unavailable, fallback plotting helpers are defined for validation purposes.

In [ ]:
import json

TEMPLATE_NOTEBOOK = Path('FIREDpy speed-Operational.ipynb')


def load_functions_from_notebook(notebook_path: Path, function_names):
    """Load selected plotting functions from a notebook by executing code cells."""
    namespace = {}
    if not notebook_path.exists():
        return namespace

    with notebook_path.open('r', encoding='utf-8') as f:
        nb = json.load(f)

    for cell in nb.get('cells', []):
        if cell.get('cell_type') == 'code':
            src = ''.join(cell.get('source', []))
            if any(f'def {fn}(' in src for fn in function_names):
                exec(src, namespace)
    return namespace


requested_functions = [
    'plot_fire_vectors_cumulative_speed',
]

plot_ns = load_functions_from_notebook(TEMPLATE_NOTEBOOK, requested_functions)
plot_ns.keys()


In [ ]:
# Fallback utility if the template function cannot be imported
def plot_fire_vectors_cumulative_speed_fallback(data: gpd.GeoDataFrame, speed_col: str, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))

    valid = data[speed_col].replace([np.inf, -np.inf], np.nan).dropna()
    if valid.empty:
        ax.text(0.5, 0.5, 'No valid speed values found', ha='center', va='center')
        ax.set_title('Cumulative Fire Spread Speed')
        return ax

    sorted_vals = np.sort(valid.values)
    cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
    ax.plot(sorted_vals, cdf, linewidth=2)
    ax.set_xlabel(speed_col)
    ax.set_ylabel('Cumulative proportion')
    ax.set_title('Cumulative Fire Spread Speed')
    return ax


## 4) Inspect available columns and identify speed/vector fields

This helps ensure we are plotting the intended new fire spread vector outputs.

In [ ]:
if gdf.empty:
    print('No data loaded.')
else:
    print(f'Rows: {len(gdf):,}')
    print('Columns:')
    print(list(gdf.columns))

# Heuristic candidates for speed/vector columns
speed_candidates = [c for c in gdf.columns if 'speed' in c.lower()] if not gdf.empty else []
vector_candidates = [c for c in gdf.columns if any(k in c.lower() for k in ['vector', 'bearing', 'azimuth', 'u_', 'v_'])] if not gdf.empty else []

print('\nSpeed candidate columns:', speed_candidates)
print('Vector candidate columns:', vector_candidates)


## 5) Validation plots

These plots are designed for quick visual checks of spread-speed behavior and vector consistency.

In [ ]:
if not gdf.empty:
    speed_col = speed_candidates[0] if speed_candidates else None

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Plot A: histogram of spread speeds
    if speed_col:
        vals = gdf[speed_col].replace([np.inf, -np.inf], np.nan).dropna()
        axes[0].hist(vals, bins=40, alpha=0.85)
        axes[0].set_title(f'Histogram of Fire Spread Speeds ({speed_col})')
        axes[0].set_xlabel(speed_col)
        axes[0].set_ylabel('Count')
    else:
        axes[0].text(0.5, 0.5, 'No speed column found', ha='center', va='center')
        axes[0].set_title('Histogram of Fire Spread Speeds')

    # Plot B: cumulative speed plot using template function if available
    if speed_col:
        template_fn = plot_ns.get('plot_fire_vectors_cumulative_speed')
        if callable(template_fn):
            try:
                template_fn(gdf)
                axes[1].remove()
            except Exception as exc:
                print(f'Template function failed, using fallback: {exc}')
                plot_fire_vectors_cumulative_speed_fallback(gdf, speed_col=speed_col, ax=axes[1])
        else:
            plot_fire_vectors_cumulative_speed_fallback(gdf, speed_col=speed_col, ax=axes[1])
    else:
        axes[1].text(0.5, 0.5, 'No speed column found', ha='center', va='center')
        axes[1].set_title('Cumulative Fire Spread Speed')

    plt.tight_layout()
    plt.show()


In [ ]:
# Optional vector direction diagnostics if direction columns are present
if not gdf.empty:
    direction_candidates = [c for c in gdf.columns if any(k in c.lower() for k in ['bearing', 'azimuth', 'direction'])]
    if direction_candidates:
        dcol = direction_candidates[0]
        vals = gdf[dcol].replace([np.inf, -np.inf], np.nan).dropna()

        fig = plt.figure(figsize=(7, 7))
        ax = fig.add_subplot(111, projection='polar')
        theta = np.deg2rad(vals % 360)
        ax.hist(theta, bins=36, alpha=0.8)
        ax.set_title(f'Fire Spread Direction Distribution ({dcol})')
        plt.show()
    else:
        print('No bearing/azimuth/direction column detected; skipped polar direction plot.')


## 6) Developer checklist

Use the plots above to check:

- Speed values are present and mostly finite.
- Distribution range is physically plausible for the study area/time period.
- Cumulative curve shape is consistent with prior known-good runs.
- Direction plot (if available) does not show obvious artifacts (e.g., all vectors snapped to a single angle).
